# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [13]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-11','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** one row is one item and the quantities of each item within that same row. I've gone ahead and changed one of the vendors in orders_json to V-11 for the sake of this lab, as I believe there was a typo.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [14]:
import pandas as pd
#normalize orders to onw row per line item

lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])

# assert the row count and the quantity total
assert len(lines) == 4
assert lines['qty'].sum() == 7
lines

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-11


### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [15]:
#left join vendor info on vendor id
joined = pd.merge(lines, pd.json_normalize(vendors_json), on='vendor_id', how='left', indicator=True)
joined

,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-11,NaN,NaN,left_only


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [16]:
#total quantity per vendor name
total_qty_per_vendor = joined.groupby('name')['qty'].sum()
total_qty_per_vendor

,qty
name,
Cav Merch,1
Hoos Burgers,3


The unmatched vendor does not appear in the table because it does not have a name in the vendor json.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [17]:
#which zone sold the most items
zone_qty = joined.groupby('zone')['qty'].sum()
zone_qty

,qty
zone,
A,4


Zone A sold the most items while 3 units are missing without a zone because it belongs to the unmatched vendor.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [18]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

processed_ragged = [
    {**record, 'lines': record.get('lines', [{}])}
    for record in ragged_json
]

second_batch = pd.json_normalize(processed_ragged, record_path='lines', meta=['order', 'vendor_id'])
second_batch

,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,NaN,NaN,5,V-10
2,Fries,NaN,6,V-01


### Q6 — Validate

In [20]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a) One row per line item is useful in this case because it creates a quantity variable which makes answering questions 3 and 4 easier, since we can look directly as the quantity rather than summing up each unit row individually.

b) a missing quantity could be any value that we do not know, while a quantity of zero is a confirmed value. Conflating the two values would bring the mean value and any related calculations down, causing inaccurate observations.